<a href="https://colab.research.google.com/github/jano87/diplomado-MLunsupervised/blob/main/AllCategoriesFullDiscoveryMLC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MercadoLibre Chile: which categories have the most buyer demand? (v4: trends + listings)

**Goal:** find which of MercadoLibre Chile's (`MLC`) ~2,800 level-3 categories attract the most buyer demand, without listing them beforehand, then group them into market profiles with PCA + k-means.

## What the API allows (tested Oct 2026 with a read-only app)
| Signal | Endpoint | Status |
|---|---|---|
| Category tree + listing counts | `/categories/{id}` | ✅ public |
| Site-wide top 50 trending searches | `/trends/MLC` | ✅ token, **refreshed weekly** |
| Top ≤50 trending searches **per category** | `/trends/MLC/{cat}` | ✅ token (404 = no public trends) |
| Keyword → category candidates | `/sites/MLC/domain_discovery/search` | ✅ public |
| Best-seller rankings `/highlights` | | ❌ 403 PolicyAgent |
| Listing search `/sites/MLC/search` | | ❌ 403 since April 2025 |

MercadoLibre doesn't expose other sellers' sales, so this notebook measures **buyer search demand** and **seller supply**, not units sold.

## How a trends list is built ([docs](https://developers.mercadolibre.cl/en_us/about-our-api/trends))
Each list of 50 is **three different rankings**, scored here as separate features:
| Positions | Segment | Meaning | Feature |
|---|---|---|---|
| 1–10 | **Fastest-growing** | highest revenue increase last week (closest thing to a sales signal) | `growth_score` |
| 11–30 | **Most wanted** | highest search volume last week | `wanted_score` |
| 31–50 | **Most popular** | biggest search rise vs. two weeks ago | `popular_score` |

## Features per level-3 category
* **`trend_depth`** (0–50): how many trending searches the category's own list has.
* **`growth_score` / `wanted_score` / `popular_score`**: keywords from the site-wide list (weight 2) and the vertical's list (weight 1) that land in this category, split by segment. A vertical's keywords are only credited to categories **inside that vertical**.
* **`log_listings`** (counts read directly from each category), plus **`listing_growth`** between refreshes.
* **`demand_vs_supply`**: trend depth above or below what the listing count predicts.
* **`keyword_turnover`**: share of the category's trending searches that are new since the previous refresh.

## When to run it
Trends refresh **once a week**. Run the notebook whenever you like (daily is fine): cell 3 checks the site-wide list first and only does the full ~2-minute collection when MercadoLibre has published new data. Otherwise it analyses the latest refresh. The log in cell 3 shows which weekday the refresh happens on.

## 0 · Credentials (one time)
In Colab's 🔑 **Secrets** panel add `ML_CLIENT_ID` (App ID) and `ML_CLIENT_SECRET` (Client Secret), with notebook access on.
Outside Colab, set the same names as environment variables, or set `ML_ACCESS_TOKEN` to an existing token.
**Never write the secret into the notebook.** This repo is public.

In [ ]:
# --- CELL 1: SETUP & DATABASE ---
import os, time, json, math, sqlite3, threading, hashlib, warnings
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
import requests
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    BASE_DIR, IN_COLAB = '/content/drive/MyDrive', True
except ImportError:
    BASE_DIR, IN_COLAB = os.environ.get('ML_BASE_DIR', '.'), False

SITE      = 'MLC'
TZ        = ZoneInfo('America/Santiago')
TODAY     = datetime.now(TZ).date().isoformat()
DB_PATH   = f'{BASE_DIR}/ml_category_demand_v3.db'      # same DB as v3: history is kept
ANALYSIS_DEPTH = 3       # level analysed (3 ≈ 2,800 categories; 2 ≈ 400 broader ones)
MAX_WORKERS    = 6       # parallel requests (well under the app's 18,000 requests/hour)
FORCE_FULL     = False   # True = collect even if MercadoLibre hasn't refreshed trends
EXCLUDE_ROOTS  = {'MLC1459', 'MLC1743', 'MLC1540'}   # real estate, vehicles, services

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.executescript("""
CREATE TABLE IF NOT EXISTS categories (
    snapshot_date TEXT, category_id TEXT, name TEXT, parent_id TEXT, root_id TEXT, l2_id TEXT,
    depth INTEGER, total_items INTEGER, path TEXT, PRIMARY KEY (snapshot_date, category_id));
CREATE TABLE IF NOT EXISTS trends (
    snapshot_date TEXT, scope TEXT, keyword TEXT, position INTEGER,
    PRIMARY KEY (snapshot_date, scope, keyword));
CREATE TABLE IF NOT EXISTS trend_status (
    snapshot_date TEXT, scope TEXT, n_keywords INTEGER, PRIMARY KEY (snapshot_date, scope));
CREATE TABLE IF NOT EXISTS keyword_candidates (keyword TEXT PRIMARY KEY, candidates TEXT, fetched_on TEXT);
CREATE TABLE IF NOT EXISTS category_paths (category_id TEXT PRIMARY KEY, path_ids TEXT);
CREATE TABLE IF NOT EXISTS refresh_log (checked_at TEXT, site_hash TEXT, changed INTEGER, snapshot_date TEXT);
CREATE TABLE IF NOT EXISTS snapshot_meta (snapshot_date TEXT PRIMARY KEY, direct_counts INTEGER);
""")
conn.commit()
DB_LOCK = threading.Lock()
print(f"DB ready: {DB_PATH} | today: {TODAY}")

In [ ]:
# --- CELL 2: API CLIENT (token, retries, thread-safe) ---
def _secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

class MeliClient:
    API = 'https://api.mercadolibre.com'

    def __init__(self, client_id=None, client_secret=None, access_token=None):
        self.client_id, self.client_secret = client_id, client_secret
        self.token, self.expires_at = access_token, (time.time() + 5 * 3600 if access_token else 0)
        self.calls, self._lock, self._local = 0, threading.Lock(), threading.local()

    def _session(self):
        if not hasattr(self._local, 's'):
            self._local.s = requests.Session(); self._local.s.headers['Accept'] = 'application/json'
        return self._local.s

    def authenticate(self):
        if not (self.client_id and self.client_secret):
            if self.token:
                return True
            raise RuntimeError("Missing ML_CLIENT_ID / ML_CLIENT_SECRET (Colab Secrets) or ML_ACCESS_TOKEN.")
        # HTTP Basic client auth (verified working); body-params as fallback
        attempts = [dict(auth=(self.client_id, self.client_secret), data={'grant_type': 'client_credentials'}),
                    dict(data={'grant_type': 'client_credentials', 'client_id': self.client_id,
                               'client_secret': self.client_secret})]
        for kw in attempts:
            r = requests.post(f'{self.API}/oauth/token', headers={'content-type': 'application/x-www-form-urlencoded',
                              'accept': 'application/json'}, timeout=30, **kw)
            if r.ok:
                d = r.json()
                self.token = d['access_token']
                self.expires_at = time.time() + int(d.get('expires_in', 21600)) - 300
                print(f"✅ Token OK (expires in {int(d.get('expires_in', 0)) // 3600} h)")
                return True
        raise RuntimeError(f"Token request failed: {r.status_code} {r.text[:200]}")

    def get(self, path, params=None, auth=True, retries=4):
        """Returns (status_code, json_or_None)."""
        if auth and time.time() > self.expires_at:
            with self._lock:
                if time.time() > self.expires_at:
                    self.authenticate()
        for attempt in range(retries):
            h = {'Authorization': f'Bearer {self.token}'} if auth else {}
            try:
                r = self._session().get(self.API + path, params=params, headers=h, timeout=30)
            except requests.RequestException:
                time.sleep(2 ** attempt); continue
            self.calls += 1
            if r.status_code == 200:
                return 200, r.json()
            if r.status_code == 401 and auth and attempt == 0:
                with self._lock:
                    self.expires_at = 0
                    self.authenticate()
                continue
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(1.5 * 2 ** attempt); continue
            return r.status_code, None
        return None, None

    def pmap(self, fn, items, workers=MAX_WORKERS, label=''):
        out, n = [], len(items)
        with ThreadPoolExecutor(workers) as ex:
            for i, res in enumerate(ex.map(fn, items), 1):
                out.append(res)
                if label and (i % 500 == 0 or i == n):
                    print(f"   {label}: {i}/{n}")
        return out

ml = MeliClient(_secret('ML_CLIENT_ID'), _secret('ML_CLIENT_SECRET'), _secret('ML_ACCESS_TOKEN'))
ml.authenticate()

# Quick probe: what is reachable today?
for name, path, auth in [('category tree', '/categories/MLC1051', False), ('root list', f'/sites/{SITE}/categories', True),
                         ('site trends', f'/trends/{SITE}', True), ('category trends', f'/trends/{SITE}/MLC1055', True),
                         ('best sellers (expected ❌)', f'/highlights/{SITE}/category/MLC1055', True)]:
    code_, _ = ml.get(path, auth=auth)
    print(f"{'✅' if code_ == 200 else '❌'} {name:<26} HTTP {code_}")

In [ ]:
# --- CELL 3: HAS MERCADOLIBRE PUBLISHED NEW TRENDS? ---
def kw_hash(kws):
    return hashlib.sha1('\n'.join(kws).encode()).hexdigest()[:16]

def site_list(day):
    return [r[0] for r in conn.execute("SELECT keyword FROM trends WHERE snapshot_date=? AND scope='SITE' "
                                       "ORDER BY position", (day,))]

def refresh_days():
    """Last day of each distinct site-wide trends list = one entry per MercadoLibre refresh."""
    days = [r[0] for r in conn.execute("SELECT DISTINCT snapshot_date FROM trend_status WHERE scope='SITE' ORDER BY 1")]
    out, prev = [], None
    for d in days:
        h = kw_hash(site_list(d))
        if out and h == prev:
            out[-1] = d                       # same data as before → keep the most recent day
        else:
            out.append(d)
        prev = h
    return out

code_, data = ml.get(f'/trends/{SITE}')
live = [t['keyword'].strip().lower() for t in data if t.get('keyword')] if code_ == 200 else None
stored_days = [r[0] for r in conn.execute("SELECT DISTINCT snapshot_date FROM trend_status WHERE scope='SITE' ORDER BY 1")]
last_day = stored_days[-1] if stored_days else None

if live is None:
    COLLECT, reason = False, f"could not read /trends (HTTP {code_}); analysing stored data"
elif last_day == TODAY:
    n_cats = conn.execute("SELECT COUNT(*) FROM categories WHERE snapshot_date=?", (TODAY,)).fetchone()[0]
    n_lists = conn.execute("SELECT COUNT(*) FROM trend_status WHERE snapshot_date=?", (TODAY,)).fetchone()[0]
    if n_cats and n_lists >= n_cats + 1:
        COLLECT, reason = False, "today's snapshot is already complete"
    else:
        COLLECT, reason = True, "today's collection is incomplete → resuming it"
elif last_day is None or FORCE_FULL or kw_hash(live) != kw_hash(site_list(last_day)):
    COLLECT, reason = True, "new trends published" if last_day else "first run"
else:
    COLLECT, reason = False, f"no refresh since {last_day}"

SNAP = TODAY if COLLECT else last_day
with DB_LOCK:
    conn.execute("INSERT INTO refresh_log VALUES (?,?,?,?)",
                 (datetime.now(TZ).isoformat(timespec='seconds'), kw_hash(live) if live else None,
                  int(reason in ('new trends published', 'first run')), SNAP))
    conn.commit()
print(f"{'🟢 COLLECT' if COLLECT else '⚪ SKIP collection'}: {reason} | analysing snapshot {SNAP}")

rd = refresh_days() + ([TODAY] if COLLECT and TODAY not in refresh_days() else [])
print("Distinct refreshes so far:", ', '.join(f"{d} ({pd.Timestamp(d).day_name()[:3]})" for d in rd))

## 1 · Discover the category tree
Read the tree down to `ANALYSIS_DEPTH`. Listing counts come **from each category's own record**; the copy embedded in the parent's child list is often stale.

In [ ]:
# --- CELL 4: CATEGORY TREE + LISTING COUNTS ---
def crawl_tree(max_depth):
    code_, roots = ml.get(f'/sites/{SITE}/categories')
    roots = [r for r in (roots or []) if r['id'] not in EXCLUDE_ROOTS]
    rows, direct, level = [], {}, [(r['id'], None, r['id'], None, 1) for r in roots]
    while level:
        results = ml.pmap(lambda x: ml.get(f'/categories/{x[0]}', auth=False)[1], level)
        nxt = []
        for (cid, parent, root, l2, depth), d in zip(level, results):
            if not d:
                continue
            direct[cid] = d.get('total_items_in_this_category')
            path = ' > '.join(p['name'].strip() for p in d.get('path_from_root', []))
            if depth == 1:
                rows.append(dict(category_id=cid, name=d['name'].strip(), parent_id=None, root_id=cid, l2_id=None,
                                 depth=1, total_items=None, path=path))
            for k in d.get('children_categories', []):
                kid_l2 = k['id'] if depth == 1 else l2
                rows.append(dict(category_id=k['id'], name=k['name'].strip(), parent_id=cid, root_id=root, l2_id=kid_l2,
                                 depth=depth + 1, total_items=None, path=f"{path} > {k['name'].strip()}"))
                if depth + 1 < max_depth:
                    nxt.append((k['id'], cid, root, kid_l2, depth + 1))
        level = nxt
    df = pd.DataFrame(rows).drop_duplicates('category_id')
    deepest = [c for c in df.category_id if c not in direct]
    counts = ml.pmap(lambda c: (c, (ml.get(f'/categories/{c}', auth=False)[1] or {}).get('total_items_in_this_category')),
                     deepest, label='direct listing counts')
    direct.update(dict(counts))
    df['total_items'] = df.category_id.map(direct)
    df.loc[df.depth == 2, 'l2_id'] = df.loc[df.depth == 2, 'category_id']
    return df.assign(snapshot_date=TODAY)

if COLLECT:
    t0 = time.time()
    tree = crawl_tree(ANALYSIS_DEPTH)
    cols = ['snapshot_date', 'category_id', 'name', 'parent_id', 'root_id', 'l2_id', 'depth', 'total_items', 'path']
    with DB_LOCK:
        conn.executemany(f"INSERT OR REPLACE INTO categories ({','.join(cols)}) VALUES ({','.join('?'*len(cols))})",
                         tree[cols].astype(object).where(tree[cols].notna(), None).values.tolist())
        conn.execute("INSERT OR REPLACE INTO snapshot_meta VALUES (?, 1)", (TODAY,))
        conn.commit()
    print(f"🌳 {len(tree):,} categories | by depth: {tree.depth.value_counts().sort_index().to_dict()} | "
          f"missing counts: {tree.total_items.isna().sum()} | {time.time()-t0:.0f}s")
else:
    tree = pd.read_sql("SELECT * FROM categories WHERE snapshot_date=?", conn, params=(SNAP,))
    print(f"Using stored tree from {SNAP}: {len(tree):,} categories")

## 2 · Collect trending searches
One call per category plus the site-wide list (skipped when there is no new refresh). A 404 means "no public trends" and is stored as 0.

In [ ]:
# --- CELL 5: TRENDS FOR EVERY CATEGORY ---
def fetch_trends(scope):
    path = f'/trends/{SITE}' if scope == 'SITE' else f'/trends/{SITE}/{scope}'
    code_, data = ml.get(path)
    if code_ == 200:
        return scope, [t['keyword'].strip().lower() for t in data if t.get('keyword')]
    return scope, ([] if code_ == 404 else None)        # None = real failure, retried on the next run

if COLLECT:
    done = {r[0] for r in conn.execute("SELECT scope FROM trend_status WHERE snapshot_date=?", (TODAY,))}
    scopes = [s for s in ['SITE'] + tree.category_id.tolist() if s not in done]      # resumable
    t0 = time.time()
    results = ml.pmap(fetch_trends, scopes, label='trend lists')
    failed = 0
    with DB_LOCK:
        for scope, kws in results:
            if kws is None:
                failed += 1; continue
            conn.execute("INSERT OR REPLACE INTO trend_status VALUES (?,?,?)", (TODAY, scope, len(kws)))
            conn.executemany("INSERT OR REPLACE INTO trends VALUES (?,?,?,?)",
                             [(TODAY, scope, k, i) for i, k in enumerate(kws, 1)])
        conn.commit()
    print(f"🔎 {len(scopes):,} lists in {time.time()-t0:.0f}s | failures: {failed} (re-run the cell to retry)")
st_ = pd.read_sql("SELECT n_keywords FROM trend_status WHERE snapshot_date=? AND scope!='SITE'", conn, params=(SNAP,))
print(f"Snapshot {SNAP}: categories with 0 trends: {(st_.n_keywords == 0).mean():.0%} | full 50: {(st_.n_keywords == 50).mean():.0%}")

In [ ]:
# --- CELL 6: KEYWORD → CATEGORY CANDIDATES (cached) ---
# domain_discovery returns up to 8 candidate categories per keyword. A keyword from a vertical's list is
# credited to the first candidate INSIDE that vertical (or dropped); site-wide keywords take the first candidate.
STOP_KEYWORDS = {'ofertas', 'oferta', 'mercado libre', 'mercadolibre', 'cyber', 'cyberday', 'cyber day',
                 'black friday', 'regalo', 'regalos', 'liquidacion', 'outlet', 'envio gratis'}

def get_candidates(kw):
    code_, res = ml.get(f'/sites/{SITE}/domain_discovery/search', params={'q': kw, 'limit': 8}, auth=False)
    return kw, ([r['category_id'] for r in res] if code_ == 200 and res else ([] if code_ == 200 else None))

def get_path(cid):
    code_, d = ml.get(f'/categories/{cid}', auth=False)
    return cid, ([p['id'] for p in d.get('path_from_root', [])] if d else None)

root_ids = tree[tree.depth == 1].category_id.tolist()
kws = pd.read_sql(f"SELECT DISTINCT keyword FROM trends WHERE snapshot_date=? AND scope IN "
                  f"({','.join('?'*(len(root_ids)+1))})", conn, params=[SNAP, 'SITE', *root_ids]).keyword
cached = {r[0] for r in conn.execute("SELECT keyword FROM keyword_candidates")}
todo = [k for k in kws if k not in cached and k not in STOP_KEYWORDS]
got = [(k, c) for k, c in ml.pmap(get_candidates, todo, label='keyword candidates') if c is not None]
with DB_LOCK:
    conn.executemany("INSERT OR REPLACE INTO keyword_candidates VALUES (?,?,?)", [(k, json.dumps(c), TODAY) for k, c in got])
    conn.commit()

known = {r[0] for r in conn.execute("SELECT category_id FROM category_paths")}
all_c = {c for (js,) in conn.execute("SELECT candidates FROM keyword_candidates") for c in json.loads(js)}
need = sorted(all_c - known)
paths = ml.pmap(get_path, need, label='category paths')
with DB_LOCK:
    conn.executemany("INSERT OR REPLACE INTO category_paths VALUES (?,?)", [(c, json.dumps(p)) for c, p in paths if p])
    conn.commit()
print(f"🗺️  {len(kws)} site/vertical keywords | {len(todo)} newly looked up | {len(need)} new paths | "
      f"{ml.calls:,} API calls so far")

## 3 · Feature engineering

In [ ]:
# --- CELL 7: FEATURES PER CATEGORY ---
SEGMENTS = [('growth', 1, 10), ('wanted', 11, 30), ('popular', 31, 50)]
def segment(pos):
    return next(name for name, lo, hi in SEGMENTS if pos <= hi) if pos <= 50 else 'popular'

def ancestor_at(path, depth):
    return path[min(depth, len(path)) - 1] if path else None

def previous_refresh(day):
    """Most recent earlier snapshot whose site-wide list differs from `day`'s (i.e. the previous refresh)."""
    h = kw_hash(site_list(day))
    earlier = [d for d in refresh_days() if d < day and kw_hash(site_list(d)) != h]
    return earlier[-1] if earlier else None

def direct_count_days():
    return {r[0] for r in conn.execute("SELECT snapshot_date FROM snapshot_meta WHERE direct_counts=1")}

def build_features(day, depth=ANALYSIS_DEPTH, site_weight=2.0):
    cats = pd.read_sql("SELECT * FROM categories WHERE snapshot_date=?", conn, params=(day,))
    status = pd.read_sql("SELECT scope, n_keywords FROM trend_status WHERE snapshot_date=?", conn, params=(day,))
    tr = pd.read_sql("SELECT scope, keyword, position FROM trends WHERE snapshot_date=?", conn, params=(day,))
    cand = dict(conn.execute("SELECT keyword, candidates FROM keyword_candidates"))
    paths = {c: json.loads(p) for c, p in conn.execute("SELECT category_id, path_ids FROM category_paths")}

    f = cats[cats.depth == depth].copy()
    names = cats.set_index('category_id')['name']
    f['root_name'], f['l2_name'] = f.root_id.map(names), f.l2_id.map(names)
    f['trend_depth'] = f.category_id.map(status.set_index('scope')['n_keywords']).fillna(0)

    # --- site + vertical keyword wins, by segment, constrained to the keyword's own vertical ---
    root_ids = set(cats[cats.depth == 1].category_id)
    t = tr[tr.scope.isin(root_ids | {'SITE'}) & ~tr.keyword.isin(STOP_KEYWORDS)].copy()
    def target(row):
        for c in json.loads(cand.get(row.keyword, '[]')):
            p = paths.get(c)
            if p and (row.scope == 'SITE' or p[0] == row.scope):
                return ancestor_at(p, depth)
        return None
    t['target'] = t.apply(target, axis=1) if len(t) else None
    t = t.dropna(subset=['target'])
    t['segment'] = t.position.map(segment)
    t['w'] = np.where(t.scope == 'SITE', site_weight, 1.0)
    for name, _, _ in SEGMENTS:
        f[f'{name}_score'] = f.category_id.map(t[t.segment == name].groupby('target')['w'].sum()).fillna(0)
    f['trend_wins'] = f[[f'{n}_score' for n, _, _ in SEGMENTS]].sum(axis=1)
    kw_list = lambda s: ', '.join(list(dict.fromkeys(s))[:3])
    f['growth_keywords'] = f.category_id.map(t[t.segment == 'growth'].sort_values('position')
                                             .groupby('target')['keyword'].apply(kw_list)).fillna('')
    f['won_keywords'] = f.category_id.map(t.sort_values('position').groupby('target')['keyword'].apply(kw_list)).fillna('')
    own = tr[tr.position <= 4].sort_values('position').groupby('scope')['keyword'].apply(', '.join)
    f['top_keywords'] = f.category_id.map(own).fillna('')

    # --- supply ---
    f['log_listings'] = np.log1p(f.total_items.fillna(0))
    mask = f.log_listings > 0
    if mask.sum() > 10:
        b, a = np.polyfit(f.loc[mask, 'log_listings'], f.loc[mask, 'trend_depth'], 1)
        f['demand_vs_supply'] = f.trend_depth - (a + b * f.log_listings)
    else:
        f['demand_vs_supply'] = 0.0

    # --- dynamics vs the previous refresh ---
    p = previous_refresh(day)
    f['listing_growth'], f['keyword_turnover'] = 0.0, np.nan
    comparable = bool(p) and {day, p} <= direct_count_days()   # pre-v4 snapshots used stale parent counts
    if comparable:
        base = pd.read_sql("SELECT category_id, total_items FROM categories WHERE snapshot_date=?", conn,
                           params=(p,)).set_index('category_id')['total_items']
        prev_n = f.category_id.map(base).replace(0, np.nan)
        f['listing_growth'] = ((f.total_items - prev_n) / prev_n).fillna(0)
    if p:
        pk = pd.read_sql("SELECT scope, keyword FROM trends WHERE snapshot_date=?", conn, params=(p,)) \
               .groupby('scope')['keyword'].apply(set)
        ck = tr.groupby('scope')['keyword'].apply(set)
        f['keyword_turnover'] = f.category_id.map(
            {c: len(ck[c] - pk.get(c, set())) / len(ck[c]) for c in ck.index if c in pk.index and len(ck[c])})
    f['keyword_turnover'] = f['keyword_turnover'].fillna(f['keyword_turnover'].median()
                                                         if f['keyword_turnover'].notna().any() else 0)
    return f, dict(previous=p, growth_comparable=comparable, n_refreshes=len([d for d in refresh_days() if d <= day]), keyword_hits=len(t))

feat, META = build_features(SNAP)
print(f"Features for {len(feat):,} level-{ANALYSIS_DEPTH} categories | snapshot {SNAP} | "
      f"previous refresh: {META['previous'] or 'none yet'} | keyword hits kept: {META['keyword_hits']}")
if META['previous'] and not META['growth_comparable']:
    print("ℹ️  listing_growth disabled: one of the two snapshots predates direct listing counts (v4).")

In [ ]:
# --- CELL 8: DEMAND SCORE & RANKING ---
WEIGHTS = {                     # percentile-ranked, so units don't matter; tweak freely
    'trend_depth':    0.35,     # breadth of buyer searches inside the category
    'growth_score':   0.20,     # fastest-growing searches (revenue increase) → closest to sales
    'wanted_score':   0.20,     # most-searched products
    'popular_score':  0.10,     # rising vs. two weeks ago
    'log_listings':   0.05,     # market size (supply)
    'listing_growth': 0.10,     # sellers entering (needs ≥ 2 refreshes)
}
w = dict(WEIGHTS)
if not META['growth_comparable']:
    w['trend_depth'] += w.pop('listing_growth')

def pct(s):                       # zeros stay at the bottom instead of sharing a mid rank
    return s.rank(pct=True, method='max').where(s > 0, 0) if (s == 0).mean() > 0.3 else s.rank(pct=True)

feat['demand_score'] = sum(pct(feat[k]) * v for k, v in w.items()) / sum(w.values()) * 100
top = feat.sort_values('demand_score', ascending=False)
fmt = {'demand_score': '{:.1f}', 'trend_depth': '{:.0f}', 'growth_score': '{:.0f}', 'wanted_score': '{:.0f}',
       'popular_score': '{:.0f}', 'total_items': '{:,.0f}', 'demand_vs_supply': '{:+.1f}', 'listing_growth': '{:+.1%}'}

show = ['root_name', 'l2_name', 'name', 'demand_score', 'trend_depth', 'growth_score', 'wanted_score',
        'popular_score', 'total_items', 'demand_vs_supply', 'top_keywords', 'won_keywords']
print(f"🔥 Top 30 categories by buyer demand (snapshot {SNAP})")
display(top[show].head(30).style.format(fmt))

print("\n🚀 Fastest-growing searches (revenue increase last week) → where they land")
display(feat[feat.growth_score > 0].sort_values(['growth_score', 'trend_depth'], ascending=False)
        [['root_name', 'name', 'growth_score', 'growth_keywords', 'trend_depth', 'total_items']].head(20).style.format(fmt))

HOT = feat.demand_score.quantile(0.90)
print(f"\n📊 Roll-up by level-2 category (top 15) | hot = demand score ≥ {HOT:.0f} (top 10%)")
l2 = (feat.groupby(['root_name', 'l2_name'])
          .agg(subcats=('name', 'size'), mean_score=('demand_score', 'mean'), max_score=('demand_score', 'max'),
               hot_subcats=('demand_score', lambda s: (s >= HOT).sum()), growth=('growth_score', 'sum'),
               wanted=('wanted_score', 'sum'), listings=('total_items', 'sum'))
          .sort_values(['hot_subcats', 'mean_score'], ascending=False))
display(l2.head(15).style.format({'mean_score': '{:.1f}', 'max_score': '{:.1f}', 'growth': '{:.0f}',
                                  'wanted': '{:.0f}', 'listings': '{:,.0f}'}))

print("\n💡 Most under-supplied demand (trend depth far above what listing count predicts)")
display(feat[feat.trend_depth >= 20].nlargest(15, 'demand_vs_supply')[['root_name', 'name', 'trend_depth', 'total_items',
        'demand_vs_supply', 'top_keywords']].style.format(fmt))

## 4 · Unsupervised profiling: PCA + k-means

In [ ]:
# --- CELL 9: PCA + K-MEANS ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

FEATURES = ['trend_depth', 'trend_wins', 'log_listings', 'demand_vs_supply', 'listing_growth', 'keyword_turnover']
sparse = [c for c in FEATURES if (feat[c] != 0).mean() < 0.05]
FEATURES = [c for c in FEATURES if feat[c].std() > 0 and c not in sparse]
if sparse:
    print(f"Left out of clustering (non-zero in <5% of categories): {sparse}")
X = feat[FEATURES].copy()
if 'trend_wins' in X:
    X['trend_wins'] = np.log1p(X['trend_wins'])
clip_cols = [c for c in X if X[c].quantile(0.99) > X[c].quantile(0.01)]
X[clip_cols] = X[clip_cols].clip(X[clip_cols].quantile(0.01), X[clip_cols].quantile(0.99), axis=1)
Xs = StandardScaler().fit_transform(X)

MIN_K, MAX_K = 4, 8
sample = np.random.RandomState(0).choice(len(Xs), min(len(Xs), 2000), replace=False)
scores = {k: silhouette_score(Xs[sample], KMeans(k, n_init=10, random_state=42).fit_predict(Xs)[sample])
          for k in range(MIN_K, MAX_K + 1)}
K = max(scores, key=scores.get)
km = KMeans(K, n_init=10, random_state=42).fit(Xs)
feat['cluster'] = km.labels_
print("Silhouette by k:", {k: round(v, 3) for k, v in scores.items()}, "→ k =", K)

prof = feat.groupby('cluster').agg(n=('name', 'size'), demand_score=('demand_score', 'mean'),
                                   trend_depth=('trend_depth', 'mean'), wins=('trend_wins', 'mean'),
                                   growth=('growth_score', 'mean'), listings=('total_items', 'median'),
                                   dvs=('demand_vs_supply', 'mean'))
order = prof.demand_score.sort_values(ascending=False).index
med_listings, mean_depth, mean_wins = feat.total_items.median(), feat.trend_depth.mean(), feat.trend_wins.mean()
names, used = {}, {}
for i, c in enumerate(order):
    p = prof.loc[c]
    if i == 0:
        n = 'High demand (site/vertical winners)'
    elif i == len(order) - 1:
        n = 'Low demand / long tail'
    elif p.listings < med_listings * 3 and p.dvs > 10:
        n = 'Under-supplied demand'
    elif p.listings > med_listings * 10 and p.trend_depth < mean_depth:
        n = 'Saturated supply'
    elif p.trend_depth >= mean_depth:
        n = 'Active demand, wins top searches' if p.wins > mean_wins * 2 else 'Active demand, broad searches'
    elif p.wins > mean_wins * 2:
        n = 'Targeted demand (few searches, but top ones)'
    else:
        n = 'Moderate demand'
    used[n] = used.get(n, 0) + 1
    names[c] = n if used[n] == 1 else f'{n} #{used[n]}'
feat['profile'] = feat.cluster.map(names)
prof.index = prof.index.map(names)
display(prof.sort_values('demand_score', ascending=False).style.format(
    {'demand_score': '{:.1f}', 'trend_depth': '{:.1f}', 'wins': '{:.2f}', 'growth': '{:.2f}', 'listings': '{:,.0f}', 'dvs': '{:+.1f}'}))

pca = PCA(2).fit(Xs); P = pca.transform(Xs)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2, 1]})
for name, g in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).groupby('profile'):
    ax[0].scatter(g.pc1, g.pc2, s=10, alpha=.55, label=f"{name} ({len(g)})")
for _, r in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).nlargest(12, 'demand_score').iterrows():
    ax[0].annotate(r['name'][:26], (r.pc1, r.pc2), fontsize=7)
ax[0].set(title=f'Category profiles (PCA, {pca.explained_variance_ratio_.sum():.0%} of variance)', xlabel='PC1', ylabel='PC2')
ax[0].legend(fontsize=8, markerscale=2)
pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2']).plot.barh(ax=ax[1], title='PCA loadings')
plt.tight_layout(); plt.show()
print(f"Data-driven check: corr(PC1, demand_score) = {abs(np.corrcoef(P[:, 0], feat.demand_score)[0, 1]):.2f}")

## 5 · Drill down to leaf niches
Fetch trends one level deeper for the top categories to find the exact niches buyers search for.

In [ ]:
# --- CELL 10: DRILL-DOWN ---
DRILL_TOP_N = 10

def children(cid):
    code_, d = ml.get(f'/categories/{cid}', auth=False)
    return [(cid, k['id'], k['name'].strip(), k.get('total_items_in_this_category')) for k in (d or {}).get('children_categories', [])]

kids = [k for ks in ml.pmap(children, top.head(DRILL_TOP_N).category_id.tolist()) for k in ks]
if kids:
    depth_by_kid = dict((s, len(k) if k is not None else 0) for s, k in ml.pmap(fetch_trends, [k[1] for k in kids]))
    niches = pd.DataFrame(kids, columns=['parent_id', 'niche_id', 'niche', 'listings'])
    niches['parent'] = niches.parent_id.map(top.set_index('category_id')['name'])
    niches['trend_depth'] = niches.niche_id.map(depth_by_kid)
    print("🎯 Niches inside the top categories")
    display(niches.sort_values(['trend_depth', 'listings'], ascending=False)
                  [['parent', 'niche', 'trend_depth', 'listings']].head(25).style.format({'listings': '{:,.0f}'}))
else:
    print("Top categories are already leaves.")

## 6 · Momentum across refreshes (needs ≥ 2 weekly refreshes)

In [ ]:
# --- CELL 11: MOMENTUM ACROSS REFRESHES ---
days = [d for d in refresh_days() if d <= SNAP][-8:]          # last 8 distinct weekly refreshes
if len(days) < 2:
    print(f"Only {len(days)} distinct refresh so far ({', '.join(days)}). Trends refresh weekly; "
          "momentum appears after the next refresh.")
else:
    hist = {d: build_features(d)[0].set_index('category_id') for d in (days[0], days[-1])}
    a, b = hist[days[0]], hist[days[-1]]
    mom = b[['root_name', 'name']].copy()
    mom['trend_depth_change'] = b.trend_depth - a.trend_depth.reindex(b.index)
    for s in ['growth_score', 'wanted_score', 'popular_score']:
        mom[s.replace('_score', '_change')] = b[s] - a[s].reindex(b.index)
    mom['wins_change'] = b.trend_wins - a.trend_wins.reindex(b.index)
    if {days[0], days[-1]} <= direct_count_days():
        mom['listing_growth'] = (b.total_items - a.total_items.reindex(b.index)) / a.total_items.reindex(b.index)
    else:
        mom['listing_growth'] = np.nan
        print("ℹ️  listing growth hidden: the oldest refresh predates direct listing counts (v4).")
    mom['keyword_turnover'] = b.keyword_turnover
    f2 = {'trend_depth_change': '{:+.0f}', 'growth_change': '{:+.0f}', 'wanted_change': '{:+.0f}',
          'popular_change': '{:+.0f}', 'wins_change': '{:+.0f}', 'listing_growth': '{:+.1%}', 'keyword_turnover': '{:.0%}'}
    print(f"📈 Rising demand ({days[0]} → {days[-1]}, {len(days)} refreshes)")
    display(mom.sort_values(['wins_change', 'trend_depth_change'], ascending=False).head(15).style.format(f2))
    print("\n📉 Falling demand")
    display(mom.sort_values(['wins_change', 'trend_depth_change']).head(10).style.format(f2))
    if mom.listing_growth.notna().any():
        print("\n🌱 Fastest supply growth (sellers entering, ≥ 500 listings)")
        display(mom[b.total_items > 500].nlargest(15, 'listing_growth').style.format(f2))

In [ ]:
# --- CELL 12: EXPORT ---
out = f'{BASE_DIR}/mlc_category_demand_{SNAP}.csv'
feat.sort_values('demand_score', ascending=False).to_csv(out, index=False)
print(f"💾 Saved {out} | API calls this run: {ml.calls:,}")

### Caveats
* **Demand ≠ sales.** Trending searches show buyer interest. The "fastest-growing" segment is based on revenue increase and is the closest signal to sales, but it's still a ranking of keywords, not units. MercadoLibre's best-seller and sales data stays blocked for this app (403); the probe in cell 2 checks it every run.
* Keyword matching uses `domain_discovery` candidates; a vertical's keyword with no candidate inside that vertical is dropped. Ambiguous words can still land in the wrong category. `STOP_KEYWORDS` removes the most generic ones.
* Each per-category list is assumed to follow the same 10/20/20 structure as the site-wide list; lists shorter than 50 are partial.
* Snapshots taken before v4 used the parent's (stale) child counts for level-3 listings, so the first `listing_growth` against a v3 snapshot mixes data sources.